In [1]:
import sys, numpy as np
import torch
print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
NumPy: 2.0.2
Torch: 2.11.0+cu128
CUDA available: True


In [2]:
!apt-get -qq update && apt-get -qq install -y ffmpeg

!pip -q install --upgrade-strategy only-if-needed \
  streamlit pyngrok \
  transformers accelerate sentence-transformers \
  pypdf arxiv langchain-text-splitters \
  soundfile streamlit-mic-recorder gTTS pillow

!python -m pip check


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.6/122.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 23.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wand

In [3]:
!pip -q install streamlit pypdf arxiv sentence-transformers langchain-text-splitters streamlit-mic-recorder soundfile ffmpeg-python
!apt-get -y install ffmpeg


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 121 not upgraded.


In [4]:
!pip -q install pymupdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 56.5 MB/s eta 0:00:00


In [5]:
%%writefile app.py
# =========================
# AI Final Project — RAG + Vision + Audio (Streamlit)
# ✅ With AUTO-MEMORY (No /remember needed)
# ✅ FIXED:
#   (A) arXiv: download FULL PDF + extract text (not only abstract)
#   (B) Not-found behavior: asks a clarifying follow-up question
# =========================

import os, re, io, pickle, time
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict, Any

import numpy as np
import pandas as pd
import streamlit as st

from pypdf import PdfReader
import arxiv
import requests

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

from streamlit_mic_recorder import mic_recorder

# Optional TTS
try:
    from gtts import gTTS
    _HAS_GTTS = True
except Exception:
    _HAS_GTTS = False


# -------------------------
# Page + Theme
# -------------------------
st.set_page_config(
    page_title="AI Final Project — RAG + Vision + Audio",
    page_icon="🤖",
    layout="wide"
)

CUSTOM_CSS = """
<style>
:root{
  --bgA:#fff7fb; --bgB:#f3e8ff; --bgC:#e6f6ff;
  --card: rgba(255,255,255,0.78);
  --text:#2a3355; --muted:#64729a;
  --accent:#c026ff; --accent2:#7cbbff;
  --border: rgba(120,120,160,0.22);
  --shadow: 0 16px 44px rgba(155,120,220,0.14);
  --magicGrad: linear-gradient(135deg, rgba(192,38,255,0.92), rgba(124,187,255,0.70));
}

/* Hide Streamlit chrome */
[data-testid="stHeader"]{display:none !important;}
#MainMenu{visibility:hidden;}
footer{visibility:hidden;}

/* Background */
html, body, .stApp{
  background:
    radial-gradient(110% 140% at 0% 0%, var(--bgA) 0%, rgba(255,247,251,0) 70%),
    radial-gradient(120% 160% at 100% 0%, var(--bgB) 0%, rgba(243,232,255,0) 65%),
    radial-gradient(160% 220% at 50% 100%, var(--bgC) 0%, rgba(230,246,255,0) 72%),
    linear-gradient(180deg, #fff9fd 0%, #eef8ff 100%);
  color: var(--text) !important;
}
.stApp, .stApp *{ color: var(--text) !important; }

h1, h2, h3, .stTitle{
  background: linear-gradient(90deg, var(--accent), var(--accent2)) !important;
  -webkit-background-clip: text !important;
  background-clip: text !important;
  -webkit-text-fill-color: transparent !important;
}

section[data-testid="stSidebar"]{
  background: linear-gradient(180deg, rgba(255,247,251,0.95) 0%, rgba(243,232,255,0.95) 100%) !important;
  border-right: 1px solid var(--border) !important;
}

div[data-testid="stAlert"], div[data-testid="metric-container"]{
  background: var(--card) !important;
  border: 1px solid var(--border) !important;
  border-radius: 16px !important;
  box-shadow: var(--shadow) !important;
}

.stButton > button{
  background: var(--magicGrad) !important;
  color: #ffffff !important;
  border: none !important;
  border-radius: 999px !important;
  box-shadow: 0 10px 24px rgba(155,120,220,0.18) !important;
}

.top-banner{
  background: linear-gradient(90deg, rgba(255,240,246,0.86), rgba(243,232,255,0.86), rgba(230,246,255,0.86));
  border: 1px solid var(--border);
  box-shadow: var(--shadow);
  border-radius: 22px;
  padding: 14px 18px;
  display:flex;
  align-items:center;
  justify-content:space-between;
  margin-bottom: 14px;
}
.brand{ font-weight: 900; font-size: 1.2rem; }
.badge { display:inline-block; padding:4px 10px; border-radius:999px; border:1px solid var(--border);
  background:rgba(255,255,255,0.55); color: var(--muted) !important; font-size: 0.85rem; }

.pastel-footer{
  position: fixed;
  left: 0; right: 0; bottom: 0;
  padding: 10px 16px;
  background: linear-gradient(90deg, rgba(255,240,246,0.92), rgba(243,232,255,0.92), rgba(230,246,255,0.92));
  border-top: 1px solid rgba(120,120,160,0.22);
  text-align: center;
  font-size: 0.92rem;
  color: rgba(100,114,154,0.95) !important;
  z-index: 9999;
}

.msg-row{ display:flex; margin: 10px 0; }
.msg-row.user{ justify-content:flex-end; }
.msg-row.bot{ justify-content:flex-start; }

.msg-bubble{
  max-width: 72%;
  padding: 12px 14px;
  border-radius: 18px;
  border: 1px solid rgba(120,120,160,0.22);
  background: rgba(255,255,255,0.80);
  box-shadow: 0 10px 26px rgba(155,120,220,0.10);
  line-height: 1.45;
  word-wrap: break-word;
}

.msg-bubble.user{
  background: linear-gradient(135deg, rgba(192,38,255,0.16), rgba(124,187,255,0.14));
}

.msg-bubble.bot{
  background: rgba(255,255,255,0.85);
}

.block-container{ padding-bottom: 80px !important; }
.small-muted{ color: rgba(100,114,154,0.95) !important; font-size: 0.92rem; }
</style>
"""
st.markdown(CUSTOM_CSS, unsafe_allow_html=True)


# -------------------------
# Config
# -------------------------
DEFAULT_LLM = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DEFAULT_EMB = "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"
DEFAULT_CAPTION_MODEL = "Salesforce/blip-image-captioning-base"
DEFAULT_WHISPER = "openai/whisper-tiny"

CHUNK_SIZE = 950
CHUNK_OVERLAP = 160

INDEX_DIR = "rag_index"
os.makedirs(INDEX_DIR, exist_ok=True)

INDEX_TEXTS_PATH = os.path.join(INDEX_DIR, "texts.pkl")
INDEX_EMBS_PATH  = os.path.join(INDEX_DIR, "embeddings.npy")
RAW_TEXT_PATH    = os.path.join(INDEX_DIR, "raw_text.txt")
CSV_META_PATH    = os.path.join(INDEX_DIR, "csv_meta.pkl")

# MEMORY persistence
MEM_NOTES_PATH   = os.path.join(INDEX_DIR, "memory_notes.pkl")
MEM_EMBS_PATH    = os.path.join(INDEX_DIR, "memory_embs.npy")
MEM_SUMMARY_PATH = os.path.join(INDEX_DIR, "memory_summary.txt")

MAX_MEMORY_NOTES = 220


# -------------------------
# Utilities
# -------------------------
def clean_text(x: str) -> str:
    return re.sub(r"\s+", " ", x).strip()

def parse_arxiv_id(user_input: str) -> str:
    s = user_input.strip()
    s = s.replace("https://arxiv.org/abs/", "").replace("http://arxiv.org/abs/", "")
    s = s.replace("https://arxiv.org/pdf/", "").replace("http://arxiv.org/pdf/", "")
    s = s.replace(".pdf", "")
    return s

def read_pdf_bytes(file_bytes: bytes, max_pages: int = 200) -> str:
    reader = PdfReader(io.BytesIO(file_bytes))
    texts = []
    for p in reader.pages[:max_pages]:
        try:
            t = p.extract_text() or ""
        except Exception:
            t = ""
        if t.strip():
            texts.append(t)
    return "\n".join(texts)

def read_csv_bytes_to_df(file_bytes: bytes, max_rows: int = 5000) -> pd.DataFrame:
    return pd.read_csv(io.BytesIO(file_bytes)).head(max_rows)

def chunk_text(text: str) -> List[str]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks = splitter.split_text(text)
    return [clean_text(c) for c in chunks if c.strip()]

def mic_bytes_to_wav16k_path(audio_bytes: bytes) -> str:
    import tempfile, subprocess
    is_wav = (len(audio_bytes) > 12 and audio_bytes[:4] == b"RIFF" and b"WAVE" in audio_bytes[:16])
    in_suffix = ".wav" if is_wav else ".webm"
    in_fd, in_path = tempfile.mkstemp(suffix=in_suffix); os.close(in_fd)
    with open(in_path, "wb") as f: f.write(audio_bytes)
    out_fd, out_path = tempfile.mkstemp(suffix=".wav"); os.close(out_fd)
    cmd = ["ffmpeg", "-y", "-i", in_path, "-ac", "1", "-ar", "16000", "-vn", out_path]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
    try: os.remove(in_path)
    except Exception: pass
    return out_path

def html_escape(s: str) -> str:
    return s.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

def render_msg(role: str, content: str):
    content = html_escape(content).replace("\n", "<br>")
    cls = "user" if role == "user" else "bot"
    st.markdown(
        f'<div class="msg-row {cls}"><div class="msg-bubble {cls}">{content}</div></div>',
        unsafe_allow_html=True
    )

def download_arxiv_pdf(pdf_url: str, timeout: int = 40) -> bytes:
    r = requests.get(pdf_url, timeout=timeout, allow_redirects=True)
    if r.status_code != 200:
        raise RuntimeError(f"PDF download failed (status={r.status_code}).")
    return r.content


# -------------------------
# In-memory Vector Index (Docs)
# -------------------------
@dataclass
class SearchHit:
    text: str
    score: float

class InMemoryVectorIndex:
    def __init__(self, emb_model: SentenceTransformer):
        self.emb_model = emb_model
        self.texts: List[str] = []
        self.embs: Optional[np.ndarray] = None

    def build(self, texts: List[str], batch_size: int = 64) -> None:
        self.texts = texts
        if not texts:
            self.embs = None
            return
        embs = self.emb_model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True
        )
        self.embs = np.asarray(embs, dtype=np.float32)

    def query_embed(self, q: str, k: int = 6) -> List[SearchHit]:
        if self.embs is None or not self.texts:
            return []
        q_emb = self.emb_model.encode([q], normalize_embeddings=True)
        q_emb = np.asarray(q_emb[0], dtype=np.float32)
        sims = self.embs @ q_emb
        k = min(k, sims.shape[0])
        if k <= 0:
            return []
        idx = np.argpartition(-sims, k-1)[:k]
        idx = idx[np.argsort(-sims[idx])]
        return [SearchHit(text=self.texts[i], score=float(sims[i])) for i in idx]

    def save(self):
        with open(INDEX_TEXTS_PATH, "wb") as f:
            pickle.dump(self.texts, f)
        if self.embs is not None:
            np.save(INDEX_EMBS_PATH, self.embs)

    def load(self) -> bool:
        if not (os.path.exists(INDEX_TEXTS_PATH) and os.path.exists(INDEX_EMBS_PATH)):
            return False
        with open(INDEX_TEXTS_PATH, "rb") as f:
            self.texts = pickle.load(f)
        self.embs = np.load(INDEX_EMBS_PATH)
        return True

    def clear_docs(self):
        self.texts = []
        self.embs = None
        for p in [INDEX_TEXTS_PATH, INDEX_EMBS_PATH, RAW_TEXT_PATH, CSV_META_PATH]:
            try:
                if os.path.exists(p):
                    os.remove(p)
            except Exception:
                pass


# -------------------------
# Cached models
# -------------------------
@st.cache_resource(show_spinner=True)
def get_embedder(model_name: str = DEFAULT_EMB) -> SentenceTransformer:
    return SentenceTransformer(model_name)

@st.cache_resource(show_spinner=True)
def get_llm(model_name: str = DEFAULT_LLM):
    use_cuda = torch.cuda.is_available()
    dtype = torch.float16 if use_cuda else torch.float32
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map="auto" if use_cuda else None,
        low_cpu_mem_usage=True
    )
    model.eval()
    return tok, model

@st.cache_resource(show_spinner=True)
def get_captioner(model_name: str = DEFAULT_CAPTION_MODEL):
    device = 0 if torch.cuda.is_available() else -1
    return pipeline("image-to-text", model=model_name, device=device)

@st.cache_resource(show_spinner=True)
def get_asr(model_name: str = DEFAULT_WHISPER):
    device = 0 if torch.cuda.is_available() else -1
    return pipeline("automatic-speech-recognition", model=model_name, device=device)


# -------------------------
# LLM helpers
# -------------------------
def build_chat_prompt(system: str, memory_summary: str, memory_notes_ctx: str, context: str,
                      history: List[Dict[str, str]], user_msg: str, tokenizer) -> str:
    messages = [{"role": "system", "content": system}]
    if memory_summary.strip():
        messages.append({"role": "system", "content": f"Conversation Memory Summary:\n{memory_summary.strip()}"})
    if memory_notes_ctx.strip():
        messages.append({"role": "system", "content": f"Saved Memory Notes:\n{memory_notes_ctx.strip()}"})
    if context.strip():
        messages.append({"role": "system", "content": f"Retrieved document context:\n{context.strip()}"})

    for m in history[-6:]:
        messages.append({"role": m["role"], "content": m["content"]})
    messages.append({"role": "user", "content": user_msg})

    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        return f"{system}\n\n{memory_summary}\n\n{memory_notes_ctx}\n\n{context}\n\n[USER]\n{user_msg}\n\n[ASSISTANT]\n"

def generate_answer(prompt: str, tokenizer, model, max_new_tokens: int = 260) -> str:
    inputs = tokenizer(prompt, return_tensors="pt")
    device = model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.10,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    gen = out[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()


# -------------------------
# Docs extractors / helpers
# -------------------------
def load_raw_text() -> str:
    if os.path.exists(RAW_TEXT_PATH):
        try:
            with open(RAW_TEXT_PATH, "r", encoding="utf-8", errors="ignore") as f:
                return f.read()
        except Exception:
            return ""
    return ""

def regex_pick(patterns: List[str], text: str) -> Optional[str]:
    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            return clean_text(m.group(1))
    return None

def answer_metadata_questions(q: str, raw_text: str) -> Optional[str]:
    ql = q.lower()

    if "title" in ql:
        title = regex_pick([r"full title:\s*(.+)", r"title:\s*(.+)", r"^\s*(.+)\s*\n"], raw_text)
        if title:
            return f"Title: {title}"

    if ("student" in ql and "name" in ql) or ("author" in ql) or ("who wrote" in ql):
        name = regex_pick([r"student name:\s*(.+)", r"author[s]?:\s*(.+)"], raw_text)
        if name:
            return f"Student/Author: {name}"

    if "student id" in ql or ("id" in ql and "student" in ql):
        sid = regex_pick([r"student id:\s*([0-9\-]+)", r"\b(\d{6,})\b"], raw_text)
        if sid:
            return f"Student ID: {sid}"

    if ql.strip() in ["summarize it", "summarize", "summary"] or ql.startswith("summarize"):
        return "__SUMMARIZE__"

    return None

def load_csv_meta() -> Dict[str, Any]:
    if os.path.exists(CSV_META_PATH):
        try:
            with open(CSV_META_PATH, "rb") as f:
                return pickle.load(f)
        except Exception:
            return {}
    return {}

def answer_csv_questions(q: str, csv_meta: Dict[str, Any]) -> Optional[str]:
    if not csv_meta:
        return None
    ql = q.lower()

    if "column" in ql or "header" in ql:
        lines = []
        for fname, meta in csv_meta.items():
            cols = meta.get("columns", [])
            lines.append(f"- {fname}: {', '.join(cols[:60])}" + (" ..." if len(cols) > 60 else ""))
        return "CSV Columns:\n" + "\n".join(lines)

    if "code name" in ql or "code names" in ql or ("codes" in ql and "name" in ql):
        lines = []
        for fname, meta in csv_meta.items():
            cols = meta.get("columns", [])
            sample = meta.get("sample", [])
            candidates = [c for c in cols if any(k in c.lower() for k in ["code", "sku", "id", "item", "product"])]
            chosen = candidates[0] if candidates else (cols[0] if cols else None)
            if chosen and sample:
                vals = [str(r.get(chosen, "")) for r in sample if str(r.get(chosen, "")).strip()]
                uniq = []
                seen = set()
                for v in vals:
                    if v not in seen:
                        uniq.append(v); seen.add(v)
                uniq = uniq[:30]
                lines.append(f"- {fname}: inferred column = '{chosen}' → sample values: {', '.join(uniq)}")
        if lines:
            return "Code Names (from CSV):\n" + "\n".join(lines)

    return None


# -------------------------
# Hybrid Retrieval (Docs)
# -------------------------
def keyword_hits(query: str, texts: List[str], k: int = 4) -> List[SearchHit]:
    toks = [t for t in re.findall(r"[A-Za-z0-9]+", query.lower()) if len(t) >= 3]
    if not toks:
        return []
    scores = []
    for i, txt in enumerate(texts):
        low = txt.lower()
        s = 0
        for tok in toks:
            s += low.count(tok)
        scores.append((s, i))
    top = [i for s, i in sorted(scores, reverse=True) if s > 0][:k]
    return [SearchHit(text=texts[i], score=float(scores[i][0])) for i in top]

def retrieve_context(index: InMemoryVectorIndex, query: str, top_k: int = 6) -> Tuple[str, List[SearchHit]]:
    emb_hits = index.query_embed(query, k=top_k)
    kw_hits = keyword_hits(query, index.texts, k=max(2, top_k // 2))

    merged = []
    seen = set()
    for h in emb_hits + kw_hits:
        key = h.text[:220]
        if key not in seen:
            merged.append(h)
            seen.add(key)

    ctx = "\n\n---\n\n".join([h.text for h in merged[:top_k]])
    return ctx, merged[:top_k]


# -------------------------
# MEMORY (Auto-save Notes + Retrieval)
# -------------------------
def load_memory_notes() -> Tuple[List[Dict[str, Any]], Optional[np.ndarray]]:
    notes = []
    embs = None
    if os.path.exists(MEM_NOTES_PATH):
        try:
            with open(MEM_NOTES_PATH, "rb") as f:
                notes = pickle.load(f)
        except Exception:
            notes = []
    if os.path.exists(MEM_EMBS_PATH):
        try:
            embs = np.load(MEM_EMBS_PATH)
        except Exception:
            embs = None
    return notes, embs

def save_memory_notes(notes: List[Dict[str, Any]], embs: Optional[np.ndarray]) -> None:
    try:
        with open(MEM_NOTES_PATH, "wb") as f:
            pickle.dump(notes, f)
    except Exception:
        pass
    try:
        if embs is None:
            if os.path.exists(MEM_EMBS_PATH):
                os.remove(MEM_EMBS_PATH)
        else:
            np.save(MEM_EMBS_PATH, embs)
    except Exception:
        pass

def normalize_note(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip().lower())

def add_memory_note(embedder: SentenceTransformer, text: str) -> bool:
    text = clean_text(text)
    if not text:
        return False

    notes, embs = load_memory_notes()
    norm = normalize_note(text)

    for n in notes[-80:]:
        if normalize_note(n.get("text", "")) == norm:
            return False

    new_id = int(time.time() * 1000)
    notes.append({"id": new_id, "text": text[:320], "ts": time.strftime("%Y-%m-%d %H:%M:%S")})

    vec = embedder.encode([text], normalize_embeddings=True)
    vec = np.asarray(vec, dtype=np.float32)

    if embs is None or (isinstance(embs, np.ndarray) and embs.size == 0):
        embs = vec
    else:
        embs = np.vstack([embs, vec])

    if len(notes) > MAX_MEMORY_NOTES:
        overflow = len(notes) - MAX_MEMORY_NOTES
        notes = notes[overflow:]
        if embs is not None and isinstance(embs, np.ndarray) and embs.shape[0] >= overflow:
            embs = embs[overflow:, :]

    save_memory_notes(notes, embs)
    return True

def delete_memory_note(note_id: int) -> None:
    notes, embs = load_memory_notes()
    if not notes:
        return
    idx = None
    for i, n in enumerate(notes):
        if int(n.get("id", 0)) == int(note_id):
            idx = i
            break
    if idx is None:
        return
    notes.pop(idx)
    if embs is not None and len(embs.shape) == 2 and embs.shape[0] > idx:
        embs = np.delete(embs, idx, axis=0)
        if embs.shape[0] == 0:
            embs = None
    save_memory_notes(notes, embs)

def clear_memory_all() -> None:
    for p in [MEM_NOTES_PATH, MEM_EMBS_PATH, MEM_SUMMARY_PATH]:
        try:
            if os.path.exists(p):
                os.remove(p)
        except Exception:
            pass

def memory_retrieve(embedder: SentenceTransformer, query: str, k: int = 3) -> Tuple[str, List[SearchHit]]:
    notes, embs = load_memory_notes()
    if not notes or embs is None or not isinstance(embs, np.ndarray) or embs.shape[0] == 0:
        return "", []

    q_vec = embedder.encode([query], normalize_embeddings=True)
    q_vec = np.asarray(q_vec[0], dtype=np.float32)
    sims = embs @ q_vec

    k = min(k, sims.shape[0])
    idx = np.argpartition(-sims, k-1)[:k]
    idx = idx[np.argsort(-sims[idx])]

    hits = []
    for i in idx:
        hits.append(SearchHit(text=notes[i]["text"], score=float(sims[i])))

    mem_ctx = "\n".join([f"- {h.text}" for h in hits[:k]])
    return mem_ctx, hits[:k]


# -------------------------
# AUTO-MEMORY Heuristics (No /remember)
# -------------------------
def looks_like_question(txt: str) -> bool:
    t = txt.strip()
    if "?" in t or "؟" in t:
        return True

    low = t.lower().strip()
    en_q = ("what", "why", "how", "when", "where", "who", "can ", "could ", "do ", "does ", "did ", "is ", "are ",
            "should ", "tell ", "explain ", "give ")


    for w in en_q:
        if low.startswith(w):
            return True
    for w in ar_q:
        if low.startswith(w):
            return True

    return False

def looks_like_memory_fact(txt: str) -> bool:
    t = clean_text(txt)
    if len(t) < 6 or len(t) > 320:
        return False
    if looks_like_question(t):
        return False

    low = t.lower()
    en_triggers = [
        "my name is", "i am ", "i'm ", "student id", "my id", "id is",
        "my email", "email is", "my phone", "phone is",
        "i live", "i'm from", "i study", "i'm a", "i work", "i am a",
        "my project", "my course", "my instructor",
        "i prefer", "i like", "i don't like", "i use"
    ]


    reject = [

        "fix", "edit", "change", "update the code", "write the code"
    ]
    if any(r in low for r in reject) and not any(tr in low for tr in (en_triggers + ar_triggers)):
        return False

    if any(tr in low for tr in en_triggers) or any(tr in t for tr in ar_triggers):
        return True

    if re.search(r"\b(name|student id|id|email|phone|university|course)\b\s*[:\-]\s*\S+", low):
        return True


    return False


# -------------------------
# Memory Summary (conversation compression)
# -------------------------
def load_memory_summary() -> str:
    if os.path.exists(MEM_SUMMARY_PATH):
        try:
            with open(MEM_SUMMARY_PATH, "r", encoding="utf-8", errors="ignore") as f:
                return f.read().strip()
        except Exception:
            return ""
    return ""

def save_memory_summary(text: str) -> None:
    try:
        with open(MEM_SUMMARY_PATH, "w", encoding="utf-8", errors="ignore") as f:
            f.write(text.strip())
    except Exception:
        pass

def maybe_update_memory_summary(tokenizer, model, min_turns: int = 18, keep_last: int = 10):
    if len(st.session_state.chat_history) < min_turns:
        return

    old = st.session_state.chat_history[:-keep_last]
    if not old:
        return

    old_text = "\n".join([f'{m["role"]}: {m["content"]}' for m in old])[:12000]
    existing = (st.session_state.memory_summary or "").strip()

    system = (
        "Write a compact conversation memory summary.\n"
        "Keep ONLY: user preferences, constraints, decisions, and stable facts.\n"
        "Do NOT invent facts. Return 6-10 bullet points."
    )
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": f"Existing summary:\n{existing}\n\nNew messages:\n{old_text}\n\nUpdated summary (bullets only):"}
    ]
    try:
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        prompt = messages[-1]["content"]

    new_sum = generate_answer(prompt, tokenizer, model, max_new_tokens=220)
    new_sum = clean_text(new_sum)
    st.session_state.memory_summary = new_sum
    save_memory_summary(new_sum)
    st.session_state.chat_history = st.session_state.chat_history[-keep_last:]


# -------------------------
# Session state
# -------------------------
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []
if "index_ready" not in st.session_state:
    st.session_state.index_ready = False
if "last_docs_note" not in st.session_state:
    st.session_state.last_docs_note = ""
if "memory_summary" not in st.session_state:
    st.session_state.memory_summary = load_memory_summary() or ""
if "auto_memory" not in st.session_state:
    st.session_state.auto_memory = True


# -------------------------
# Sidebar
# -------------------------
st.sidebar.markdown("### 🤖 AI Final Project")
st.sidebar.markdown(
    '<span class="badge">RAG</span> <span class="badge">Vision</span> <span class="badge">Audio</span> <span class="badge">Memory</span>',
    unsafe_allow_html=True
)

mode = st.sidebar.radio(
    "Choose mode:",
    ["💬 Chat (RAG QA)", "📝 Summarize Document", "🖼️ Image Captioning", "📊 CSV Explorer", "🧠 Memory", "⚙️ Settings"],
    index=0
)

st.sidebar.markdown("---")
st.sidebar.markdown("#### 📁 Upload Documents (PDF / CSV)")
uploaded_files = st.sidebar.file_uploader(
    "Upload PDF or CSV files (multiple allowed):",
    type=["pdf", "csv"],
    accept_multiple_files=True
)

st.sidebar.markdown("#### 📚 arXiv Paper")
arxiv_input = st.sidebar.text_input("Enter arXiv link or ID (e.g., 1706.03762):")

colA, colB = st.sidebar.columns(2)
with colA:
    build_btn = st.button("🧠 Build / Update Index", type="primary")
with colB:
    clear_btn = st.button("🧹 Clear Docs Index")

st.sidebar.markdown("---")
audio_in = st.sidebar.toggle("🎙️ Audio Input (STT)", value=True)
audio_out = st.sidebar.toggle("🔊 Read Answer Aloud (TTS)", value=False)

st.sidebar.markdown("---")
debug_show_chunks = st.sidebar.checkbox("🧪 Debug: show retrieved chunks", value=False)

st.sidebar.markdown("---")
st.session_state.auto_memory = st.sidebar.toggle("🧠 Auto-Memory (save facts automatically)", value=st.session_state.auto_memory)
st.sidebar.caption("Auto-Memory saves facts/preferences automatically (not questions).")

st.sidebar.markdown("---")
if st.sidebar.button("🗑️ Clear Chat"):
    st.session_state.chat_history = []
    st.rerun()


# -------------------------
# Load models + index
# -------------------------
embedder = get_embedder(DEFAULT_EMB)
tokenizer, llm = get_llm(DEFAULT_LLM)

index = InMemoryVectorIndex(embedder)
st.session_state.index_ready = index.load()


# -------------------------
# Index build logic
# -------------------------
def build_index_from_inputs(files, arxiv_str: str, max_pages: int = 200) -> Tuple[bool, str, int]:
    texts_all = []
    raw_parts = []
    csv_meta: Dict[str, Any] = {}

    for uf in files or []:
        b = uf.getvalue()
        name = uf.name

        if name.lower().endswith(".pdf"):
            t = read_pdf_bytes(b, max_pages=max_pages)
            if t.strip():
                raw_parts.append(f"[PDF:{name}]\n{t}")
                texts_all.append(f"[PDF:{name}]\n{t}")

        elif name.lower().endswith(".csv"):
            try:
                df = read_csv_bytes_to_df(b, max_rows=5000)
                csv_meta[name] = {
                    "columns": list(df.columns),
                    "sample": df.head(60).to_dict(orient="records")
                }
                preview = df.head(120).to_string(index=False)
                raw_parts.append(f"[CSV:{name}]\nCOLUMNS: {', '.join(df.columns)}\n\nPREVIEW:\n{preview}")
                texts_all.append(f"[CSV:{name}]\nCOLUMNS: {', '.join(df.columns)}\n\nPREVIEW:\n{preview}")
            except Exception as e:
                return False, f"Failed to read CSV {name}: {e}", 0

    # ✅ arXiv: ABSTRACT + FULL PDF TEXT
    if arxiv_str and arxiv_str.strip():
        try:
            arx_id = parse_arxiv_id(arxiv_str)
            search = arxiv.Search(id_list=[arx_id])
            paper = next(search.results())

            abs_txt = paper.summary or ""
            block_abs = f"[ARXIV:{arx_id}] Title: {paper.title}\nAbstract:\n{abs_txt}"
            raw_parts.append(block_abs)
            texts_all.append(block_abs)

            pdf_url = getattr(paper, "pdf_url", None)
            if pdf_url:
                with st.spinner("Downloading arXiv PDF and extracting text..."):
                    pdf_bytes = download_arxiv_pdf(pdf_url)
                    pdf_text = read_pdf_bytes(pdf_bytes, max_pages=max_pages)

                if pdf_text.strip():
                    block_pdf = f"[ARXIV_PDF:{arx_id}] Title: {paper.title}\n\n{pdf_text}"
                    raw_parts.append(block_pdf)
                    texts_all.append(block_pdf)
            else:
                raw_parts.append(f"[ARXIV:{arx_id}] (No pdf_url found by arxiv library)")

        except Exception as e:
            return False, f"Failed to fetch arXiv paper/PDF: {e}", 0

    if not texts_all:
        return False, "No content to index. Upload PDF/CSV or enter an arXiv ID.", 0

    big_text = "\n\n".join(texts_all)
    chunks = chunk_text(big_text)
    if not chunks:
        return False, "Text extracted but no readable chunks. The PDF might be scanned images.", 0

    chunks = chunks[:2500]

    with st.spinner("Embedding + building index..."):
        index.build(chunks, batch_size=64)
        index.save()

    try:
        raw_text = "\n\n".join(raw_parts)
        with open(RAW_TEXT_PATH, "w", encoding="utf-8", errors="ignore") as f:
            f.write(raw_text[:600000])
    except Exception:
        pass

    try:
        with open(CSV_META_PATH, "wb") as f:
            pickle.dump(csv_meta, f)
    except Exception:
        pass

    return True, f"✅ Index built with {len(chunks)} chunks.", len(chunks)


if clear_btn:
    index.clear_docs()
    st.session_state.index_ready = False
    st.session_state.last_docs_note = "✅ Docs index cleared. (Memory kept)"
    st.rerun()

if build_btn:
    ok, note, n = build_index_from_inputs(uploaded_files, arxiv_input, max_pages=200)
    st.session_state.index_ready = ok
    st.session_state.last_docs_note = note
    st.rerun()


# -------------------------
# Top banner
# -------------------------
st.markdown(
    """
    <div class="top-banner">
      <div>
        <div class="brand">🌸 AI Final Project Assistant</div>
        <div class="small-muted">RAG + Vision + Voice + Auto-Memory</div>
      </div>
      <div style="font-size:1.25rem; opacity:0.9;">🌷 🌼 🌸</div>
    </div>
    """,
    unsafe_allow_html=True
)

st.title("🤖 LLM Chatbot (RAG + Vision + Audio + Auto-Memory)")
if st.session_state.last_docs_note:
    st.info(st.session_state.last_docs_note)


# -------------------------
# Mode: Image Captioning
# -------------------------
if mode == "🖼️ Image Captioning":
    st.subheader("🖼️ Multimodal Task: Image Captioning (BLIP)")
    img = st.file_uploader("Upload an image:", type=["png", "jpg", "jpeg", "webp"])
    if img is not None:
        from PIL import Image
        pil = Image.open(img).convert("RGB")
        st.image(pil, caption="Uploaded Image", use_container_width=True)

        if st.button("✨ Generate Caption", type="primary"):
            with st.spinner("Generating caption..."):
                cap = get_captioner(DEFAULT_CAPTION_MODEL)
                out = cap(pil, max_new_tokens=60)
                caption = out[0]["generated_text"] if out else ""
            st.success(caption if caption else "Could not generate a caption.")
    st.stop()


# -------------------------
# Mode: CSV Explorer
# -------------------------
if mode == "📊 CSV Explorer":
    st.subheader("📊 CSV Explorer")
    csv_meta = load_csv_meta()
    if not csv_meta:
        st.warning("No CSV cached yet. Upload CSV files and click Build / Update Index.")
        st.stop()

    for fname, meta in csv_meta.items():
        st.markdown(f"### {fname}")
        st.write("Columns:", meta.get("columns", []))
        sample = meta.get("sample", [])
        if sample:
            st.dataframe(pd.DataFrame(sample).head(30), use_container_width=True)
    st.stop()


# -------------------------
# Mode: Summarize Document
# -------------------------
if mode == "📝 Summarize Document":
    st.subheader("📝 Summarize")
    st.caption("Upload files + Build Index, then summarize from extracted raw text.")

    if st.button("🧾 Summarize Now", type="primary"):
        raw = load_raw_text()
        if not raw.strip():
            st.error("No raw text found. Upload files and click Build / Update Index.")
        else:
            prompt = (
                "Summarize the following content in English.\n"
                "Rules:\n"
                "- Use 8-12 bullet points\n"
                "- Then add 2-line conclusion\n"
                "- Do NOT invent numbers or facts\n\n"
                f"{raw[:22000]}\n\nSummary:"
            )
            with st.spinner("Summarizing..."):
                summ = generate_answer(prompt, tokenizer, llm, max_new_tokens=260)
            st.success("✅ Done")
            st.markdown(summ)
    st.stop()


# -------------------------
# Mode: Memory
# -------------------------
if mode == "🧠 Memory":
    st.subheader("🧠 Memory (Auto-Saved)")
    st.caption("This memory auto-saves facts/preferences (not questions).")

    st.markdown("#### 🧾 Memory Summary (auto)")
    st.text_area(" ", value=st.session_state.memory_summary if st.session_state.memory_summary else "No summary yet.", height=120, disabled=True)

    colS1, colS2 = st.columns(2)
    with colS1:
        if st.button("🧠 Update Summary", type="primary"):
            maybe_update_memory_summary(tokenizer, llm)
            st.success("Summary updated.")
            st.rerun()
    with colS2:
        if st.button("🧽 Clear Summary"):
            st.session_state.memory_summary = ""
            save_memory_summary("")
            st.rerun()

    st.divider()

    notes, _ = load_memory_notes()
    st.markdown("#### 📌 Saved Notes")
    if not notes:
        st.info("No saved notes yet.")
    else:
        with st.expander("Show notes", expanded=True):
            for n in notes[-40:][::-1]:
                st.write(f"• ({n.get('ts','')}) {n.get('text','')}")

        options = [(str(n["id"]), n["text"][:60] + ("..." if len(n["text"]) > 60 else "")) for n in notes]
        sel = st.selectbox("Delete a note:", options=[o[0] for o in options], format_func=lambda x: dict(options).get(x, x))
        if st.button("🗑️ Delete Selected"):
            delete_memory_note(int(sel))
            st.success("Deleted.")
            st.rerun()

    st.divider()
    if st.button("🔥 Clear ALL Memory (Notes + Summary)"):
        clear_memory_all()
        st.session_state.memory_summary = ""
        st.success("All memory cleared.")
        st.rerun()

    st.stop()


# -------------------------
# Mode: Settings
# -------------------------
if mode == "⚙️ Settings":
    st.subheader("⚙️ Settings")
    st.write("This app is optimized for accuracy with Hybrid Retrieval + Extractors + Auto-Memory.")
    st.info("Tip: For best RAG accuracy, always click Build / Update Index after uploading files.")
    st.stop()


# -------------------------
# Chat (RAG QA + Auto-Memory)
# -------------------------
NOT_FOUND_MSG = (
    "I couldn't find this in the provided documents or memory.\n\n"
    "Could you tell me which file/document I should use, or upload the document that contains this information?"
)

SYSTEM_MSG = (
    "You are a strict assistant.\n"
    "You may answer ONLY using:\n"
    "1) Retrieved document context, and/or\n"
    "2) Saved Memory Notes.\n"
    "If there is not enough evidence, say you can't find it and ask a short clarifying question.\n"
    "When reporting numbers/metrics, copy them EXACTLY from evidence; do not estimate."
)

for m in st.session_state.chat_history:
    render_msg(m["role"], m["content"])


# Audio input
user_text = None
if audio_in:
    st.markdown("#### 🎙️ Audio Input")
    audio = mic_recorder(
        start_prompt="🎙️ Start recording",
        stop_prompt="⏹️ Stop",
        just_once=True,
        use_container_width=True
    )
    if audio and "bytes" in audio and audio["bytes"]:
        asr = get_asr(DEFAULT_WHISPER)
        wav_path = mic_bytes_to_wav16k_path(audio["bytes"])
        with st.spinner("Transcribing (Whisper)..."):
            result = asr(wav_path)
            user_text = (result.get("text") or "").strip()
        try:
            os.remove(wav_path)
        except Exception:
            pass
        if user_text:
            st.caption(f"Transcribed: {user_text}")

typed = st.chat_input("Type your question here (or use the microphone)...")
if typed:
    user_text = typed.strip()


if user_text:
    st.session_state.chat_history.append({"role": "user", "content": user_text})
    render_msg("user", user_text)

    # Auto-memory
    auto_saved = False
    if st.session_state.auto_memory and looks_like_memory_fact(user_text):
        auto_saved = add_memory_note(embedder, user_text)

    raw_text = load_raw_text()
    csv_meta = load_csv_meta()

    # retrieve memory relevant to question
    mem_ctx, mem_hits = memory_retrieve(embedder, user_text, k=3)

    # 1) docs metadata extractor
    meta_ans = answer_metadata_questions(user_text, raw_text)
    if meta_ans == "__SUMMARIZE__":
        if not raw_text.strip():
            ans = NOT_FOUND_MSG
        else:
            prompt = (
                "Summarize the following content in English.\n"
                "Rules:\n"
                "- Use 8-12 bullet points\n"
                "- Then add 2-line conclusion\n"
                "- Do NOT invent numbers or facts\n\n"
                f"{raw_text[:22000]}\n\nSummary:"
            )
            with st.spinner("Summarizing..."):
                ans = generate_answer(prompt, tokenizer, llm, max_new_tokens=260)

    elif meta_ans:
        ans = meta_ans

    else:
        # 2) CSV extractor
        csv_ans = answer_csv_questions(user_text, csv_meta)
        if csv_ans:
            ans = csv_ans
        else:
            # 3) Docs retrieval
            context = ""
            hits = []
            if st.session_state.index_ready:
                context, hits = retrieve_context(index, user_text, top_k=6)

            evidence_ok = bool(context.strip()) or bool(mem_ctx.strip())

            if debug_show_chunks:
                with st.expander("🔎 Evidence (debug)", expanded=True):
                    st.write(f"Docs index ready: {st.session_state.index_ready}")
                    st.write(f"Auto-saved memory this turn: {auto_saved}")
                    st.write(f"Memory hits: {len(mem_hits)}")
                    if mem_hits:
                        st.markdown("**Memory hits:**")
                        for i, h in enumerate(mem_hits, 1):
                            st.write(f"{i}) score={h.score:.3f} | {h.text}")
                    if hits:
                        st.markdown("**Document chunks:**")
                        for i, h in enumerate(hits, 1):
                            st.markdown(f"**Chunk {i}**")
                            st.write(h.text[:900] + ("..." if len(h.text) > 900 else ""))
                            st.divider()

            if not evidence_ok:
                ans = NOT_FOUND_MSG
            else:
                prompt = build_chat_prompt(
                    system=SYSTEM_MSG,
                    memory_summary=st.session_state.memory_summary,
                    memory_notes_ctx=mem_ctx,
                    context=context,
                    history=st.session_state.chat_history,
                    user_msg=user_text,
                    tokenizer=tokenizer
                )
                with st.spinner("Thinking..."):
                    ans = generate_answer(prompt, tokenizer, llm, max_new_tokens=260)

                # If model tries to answer with weak evidence, enforce clarify
                if (len(context.strip()) < 30) and (len(mem_ctx.strip()) < 10):
                    ans = NOT_FOUND_MSG

    if auto_saved:
        ans = ans + "\n\n_(Saved to memory.)_"

    st.session_state.chat_history.append({"role": "assistant", "content": ans})
    render_msg("assistant", ans)

    maybe_update_memory_summary(tokenizer, llm)

    if audio_out and _HAS_GTTS and ans.strip():
        try:
            tts = gTTS(ans, lang="en")
            mp3_fp = io.BytesIO()
            tts.write_to_fp(mp3_fp)
            mp3_fp.seek(0)
            st.audio(mp3_fp.read(), format="audio/mp3")
        except Exception:
            st.caption("TTS failed (gTTS).")

    st.rerun()

st.markdown('<div class="pastel-footer">🌸 AI Final Project Assistant • RAG + Vision + Voice + Auto-Memory</div>', unsafe_allow_html=True)


Writing app.py


In [6]:
import os, time, subprocess
from pyngrok import ngrok
import getpass

# 1) حطي ngrok token بشكل آمن
token = getpass.getpass("Paste your ngrok authtoken: ").strip()
ngrok.set_auth_token(token)

# 2) اقفلي أي تشغيل قديم
try: ngrok.kill()
except: pass
subprocess.run("pkill -f streamlit", shell=True, check=False)

# 3) شغلي Streamlit
!nohup streamlit run app.py --server.port 8501 --server.address 0.0.0.0 \
  --server.enableCORS false --server.enableXsrfProtection false \
  --server.fileWatcherType none > streamlit.log 2>&1 &

time.sleep(2)

# 4) افتحي Tunnel
public_url = ngrok.connect(8501, "http").public_url
print("✅ Streamlit URL:", public_url)

!tail -n 50 streamlit.log


Paste your ngrok authtoken: ··········
✅ Streamlit URL: https://f402-35-203-157-232.ngrok-free.app


2026-07-03 02:29:21.573 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.203.157.232:8501

